# B5 單字錯誤分析（Colab GPU）

1. 上方選單「執行階段 → 變更執行階段類型」選 **GPU**（T4 就夠；有 Pro 可選 L4／A100 比較快）
2. 「執行階段 → 全部執行」
3. 第 3 步會請你登入 Google：用 Firebase 專案（eng-3385e）擁有者的帳號，分析完一個單元就直接更新網站，不用下載、上傳
4. 想讓它自己顧著：最後一格「持續自動分析」會每幾分鐘檢查一次，老師改了題目就自動分析並更新（Colab 分頁要開著）

單字表（大考中心 Level 1–6）會自動從 Hugging Face 的 `nhmccyj/b5-wordbank` 下載；要換成別的單字表，就把 CSV 放進這個資料夾的 `wordbank` 子資料夾。

In [ ]:
#@title 1. 連接雲端硬碟、準備程式
import os, sys, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
FOLDER_NAME = 'B5 錯誤分析 Colab'  #@param {type:"string"}
cands = glob.glob('/content/drive/MyDrive/**/' + FOLDER_NAME, recursive=True) or glob.glob('/content/drive/MyDrive/' + FOLDER_NAME)
assert cands, '在雲端硬碟找不到資料夾「' + FOLDER_NAME + '」'
SRC = cands[0]
WORK = '/content/brain'
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns('out', '*.ipynb'))
sys.path.insert(0, WORK)
os.chdir(WORK)
# 單字表：資料夾裡的 wordbank/ 沒有的話，從 Hugging Face（nhmccyj/b5-wordbank，大考中心詞彙表 Level 1-6）下載
import urllib.request, urllib.parse
BANK = os.path.join(WORK, 'wordbank'); os.makedirs(BANK, exist_ok=True)
bank = [f for f in os.listdir(BANK) if f.lower().endswith(('.csv', '.tsv', '.txt'))]
if not bank:
    name = '大考中心詞彙表_Level1-6.csv'
    url = 'https://huggingface.co/datasets/nhmccyj/b5-wordbank/resolve/main/' + urllib.parse.quote(name)
    urllib.request.urlretrieve(url, os.path.join(BANK, name))
    bank = [name]
print('程式資料夾：', SRC)
print('單字表：', bank)
!nvidia-smi -L || echo '⚠ 沒有 GPU：請到「執行階段 → 變更執行階段類型」選 GPU，再全部執行'

In [ ]:
#@title 2. 安裝套件（第一次約 1 分鐘）
!pip -q install nltk==3.10.3 jieba==0.42.1 lemminflect transformers accelerate
import nltk
for p in ('wordnet', 'cmudict', 'averaged_perceptron_tagger_eng'):
    nltk.download(p, download_dir=os.path.join(WORK, 'nltk_data'), quiet=True)
import torch
print('GPU：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '沒有（會很慢）')

In [ ]:
#@title 3. 登入 Google（用你的帳號直接把結果寫到網站；不用存任何金鑰）
from google.colab import auth
auth.authenticate_user()
import google.auth, google.auth.transport.requests
_creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/datastore'])
def token():
    if not _creds.valid:
        _creds.refresh(google.auth.transport.requests.Request())
    return _creds.token
print('已登入，可以直接更新網站')

In [ ]:
#@title 4. 準備：讀網站上的單字單元、分析、寫回網站
import json, random, time, urllib.request, urllib.parse
import brain
PROJECT = 'eng-3385e'
ROOT = f'projects/{PROJECT}/databases/(default)/documents'
FS = f'https://firestore.googleapis.com/v1/{ROOT}/public/'
def public(doc_id):
    try:
        raw = json.loads(urllib.request.urlopen(FS + urllib.parse.quote(doc_id), timeout=30).read())
        return json.loads(raw['fields']['json']['stringValue'])
    except urllib.error.HTTPError as e:
        if e.code == 404: return None
        raise
def pending(analyze_all=False, only=()):
    """回傳 [(單元代號, 標題, 題目資料, 狀態)]：題目改過或沒分析過的（或指定的）"""
    cfg = public('config')
    titles = {u['id']: u.get('title', u['id']) for u in cfg.get('customUnits', [])}
    types = {u['id']: u['type'] for u in cfg.get('customUnits', [])}
    out = []
    for uid in cfg.get('content', {}):
        if types.get(uid) != 'vocab' or (only and uid not in only):
            continue
        c = public('content_' + uid)
        data = (c or {}).get('data') or {}
        words = [w for w in data.get('words', []) if w.get('word')]
        if not words:
            continue
        old = public('analysis_' + uid)
        state = '已分析' if old and old.get('sig') == brain.signature(words) else ('題目改過' if old else '還沒分析')
        if analyze_all or only or state != '已分析':
            out.append((uid, titles.get(uid, uid), data, state))
    return out
def analyze(uid, data):
    a = brain.analyze(data)
    a['unsafe'] = brain.option_safety(data, a)
    a['extra'] = brain.extra_options(data, a, random.Random(uid), uid=uid)
    a['at'] = int(time.time() * 1000)
    return a
def publish(uid, a):
    """寫到 public/analysis_<單元>（和老師後台「開始分析」存的地方、格式一樣）"""
    body = {'writes': [{'update': {'name': f'{ROOT}/public/analysis_{uid}', 'fields': {'json': {'stringValue': json.dumps(a, ensure_ascii=False)}}}}]}
    req = urllib.request.Request(f'https://firestore.googleapis.com/v1/{ROOT}:commit', data=json.dumps(body).encode(), method='POST',
                                 headers={'Authorization': 'Bearer ' + token(), 'Content-Type': 'application/json'})
    try:
        urllib.request.urlopen(req, timeout=60).read()
    except urllib.error.HTTPError as e:
        msg = e.read().decode()[:300]
        if e.code in (401, 403):
            raise RuntimeError('這個 Google 帳號沒有權限寫入網站資料庫：請用 Firebase 專案 eng-3385e 的擁有者帳號登入，'
                               '或到 Google Cloud Console → IAM 幫這個帳號加上「Cloud Datastore 使用者」角色。' + msg)
        raise RuntimeError(f'寫入失敗（HTTP {e.code}）：{msg}')
OUT = os.path.join(SRC, 'out'); os.makedirs(OUT, exist_ok=True)
def run(items):
    done = []
    for i, (uid, title, data, state) in enumerate(items, 1):
        t0 = time.time()
        print(f'[{i}/{len(items)}] {title}（{state}）…', end=' ', flush=True)
        a = analyze(uid, data)
        with open(os.path.join(OUT, uid + '.json'), 'w', encoding='utf-8') as f:
            json.dump(a, f, ensure_ascii=False)
        publish(uid, a)
        done.append(uid)
        print(f'{len(a["words"])} 個字，{time.time() - t0:.0f} 秒，已更新網站')
    return done
print('準備好了')

In [ ]:
#@title 5. 現在分析一次（分析完一個單元就馬上更新網站）
ANALYZE_ALL = False  #@param {type:"boolean"}
ONLY_UNITS = ''  #@param {type:"string"}
only = [x.strip() for x in ONLY_UNITS.split(',') if x.strip()]
items = pending(ANALYZE_ALL, only)
for uid, title, data, state in items:
    print('➜', title, f'({uid})', len([w for w in data.get('words', []) if w.get('word')]), '個字', state)
if not items:
    print('全部都分析好了（要重做就勾 ANALYZE_ALL，或在 ONLY_UNITS 填單元代號）')
else:
    print()
    done = run(items)
    print('完成', len(done), '個單元；學生重新打開單元就會用到新的分析')

In [ ]:
#@title 6.（選用）持續自動分析：老師改了題目，幾分鐘內自動分析並更新網站
#@markdown 這格會一直執行，Colab 分頁開著就會持續檢查；要停就按這格左邊的停止鍵
CHECK_EVERY_MINUTES = 3  #@param {type:"integer"}
import datetime
print('開始持續檢查，每', CHECK_EVERY_MINUTES, '分鐘一次')
while True:
    try:
        items = pending()
        if items:
            print(datetime.datetime.now().strftime('%H:%M'), '發現', len(items), '個單元要分析')
            run(items)
    except Exception as e:
        print(datetime.datetime.now().strftime('%H:%M'), '這次失敗，稍後再試：', e)
    time.sleep(max(1, CHECK_EVERY_MINUTES) * 60)